# Clustering

## 1. Librerías y configuraciones

In [2]:

import json
import random
from collections import Counter
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TripletEvaluator

In [3]:
# CONFIGURACIÓN GLOBAL
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = Path("2_Modelos/clustering")
DATA_PATH = "1_Data/processed/clustering_data_cleaned.csv"


In [5]:
config = {
    'batch_size': 16,
    'num_epochs': 3,
    'learning_rate': 2e-5,
    'warmup_steps': 100,
    'evaluation_steps': 50,  # Evaluar cada 50 steps
    'save_steps': 100,
    'max_seq_length': 256,
    'min_cluster_size': 5,
    'min_samples': 3,
    'val_split': 0.2,
    'random_seed': 42
}

In [4]:
# Inicialización
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# Set seed para reproducibilidad
random.seed(config['random_seed'])
np.random.seed(config['random_seed'])
torch.manual_seed(config['random_seed'])

In [7]:
# Cargar modelo base
print(f"Usando device: {device}")
print(f"Cargando modelo base: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
model.to(device)


Usando device: cpu
Cargando modelo base: sentence-transformers/all-MiniLM-L6-v2


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [8]:
# Log de configuración
print("\n" + "="*60)
print("CONFIGURACIÓN DE CLUSTERING")
print("="*60)
for key, value in config.items():
    print(f"{key}: {value}")
print("="*60 + "\n")



CONFIGURACIÓN DE CLUSTERING
batch_size: 16
num_epochs: 3
learning_rate: 2e-05
warmup_steps: 100
evaluation_steps: 50
save_steps: 100
max_seq_length: 256
min_cluster_size: 5
min_samples: 3
val_split: 0.2
random_seed: 42



## 2. Carga y preparación de datos

In [9]:
df = pd.read_csv(DATA_PATH)

In [10]:
required_cols = ['text', 'title', 'source']
df = df.dropna(subset=required_cols)
print(f"Cargados y limpiados {len(df)} documentos.")

Cargados y limpiados 434 documentos.


### Pares de entrenamiento

In [11]:

all_examples = []
source_groups = df.groupby('source')['text'].apply(list).to_dict()

for source, texts in source_groups.items():
    if len(texts) < 3:
        continue
    for i in range(len(texts)):
        for j in range(i+1, min(i+4, len(texts))): # Max 3 pares por texto
            example = InputExample(
                texts=[texts[i], texts[j]],
                label=0.9
            )
            all_examples.append(example)


### Train/val split

In [12]:
train_size = int(len(all_examples) * (1 - config['val_split']))
train_examples = all_examples[:train_size]
val_examples = all_examples[train_size:]

print(f"✓ Pares de entrenamiento: {len(train_examples)}")
print(f"✓ Pares de validación: {len(val_examples)}")


✓ Pares de entrenamiento: 621
✓ Pares de validación: 156


## 3. Creación del Triplet Evaluator

In [13]:

# Crear datos para TripletEvaluator (anchor, positive, negative)
source_to_texts = df.groupby('source')['text'].apply(list).to_dict()
sources = list(source_to_texts.keys())

anchors, positives, negatives = [], [], []

for source in sources:
    texts = source_to_texts[source]
    if len(texts) < 2:
        continue
    
    for i in range(min(5, len(texts))): # Max 5 triplets por fuente
        if i + 1 >= len(texts):
            break
        
        anchor = texts[i]
        positive = texts[i + 1]
        
        # Seleccionar negative de otra fuente
        other_sources = [s for s in sources if s != source]
        if not other_sources:
            continue
        
        neg_source = random.choice(other_sources)
        # Asegurar que la fuente negativa tiene textos
        if not source_to_texts[neg_source]:
             continue
        
        negative = random.choice(source_to_texts[neg_source])
        
        anchors.append(anchor)
        positives.append(positive)
        negatives.append(negative)


In [14]:
if len(anchors) == 0:
    print("⚠ No se pudieron crear triplets suficientes. Usando evaluador Dummy.")
    # Si no hay suficientes datos para triplets reales
    evaluator = None 
else:
    print(f"✓ Creados {len(anchors)} triplets para evaluación")
    evaluator = TripletEvaluator(
        anchors=anchors,
        positives=positives,
        negatives=negatives,
        name='validation',
        show_progress_bar=True
    )


✓ Creados 212 triplets para evaluación


## 4. Fine-Tuning del modelo (Contrastive Learning)

In [16]:

# DataLoader
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=config['batch_size']
)

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model)

# Calcular steps
num_train_steps = len(train_dataloader) * config['num_epochs']
warmup_steps = min(config['warmup_steps'], num_train_steps // 10)

print(f"Pasos totales: {num_train_steps}")
print(f"Warmup steps: {warmup_steps}")


Pasos totales: 117
Warmup steps: 11


In [17]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=config['num_epochs'],
    warmup_steps=warmup_steps,
    output_path=str(OUTPUT_DIR / "fine_tuned_model"),
    show_progress_bar=True,
    evaluation_steps=config['evaluation_steps'],
    save_best_model=True, # Guarda el mejor modelo según TripletEvaluator
    optimizer_params={'lr': config['learning_rate']},
)

print("\n✓ Fine-tuning completado! Modelo guardado en 'fine_tuned_model'")

# Cargar el mejor modelo (fine-tuned) para las siguientes etapas
print("\nCargando mejor modelo fine-tuned...")
model = SentenceTransformer(
    str(OUTPUT_DIR / "fine_tuned_model")
)
model.to(device)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/arielamishaancohen/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Validation Cosine Accuracy
39,No log,No log,0.754717
50,No log,No log,0.787736
78,No log,No log,0.773585
100,No log,No log,0.773585
117,No log,No log,0.778302


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]


✓ Fine-tuning completado! Modelo guardado en 'fine_tuned_model'

[PASO 5/7] Cargando mejor modelo fine-tuned...


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [18]:
# Guardar configuración final (incluyendo el caso de no fine-tuning)
config_to_save = config.copy()
config_to_save['model_name'] = MODEL_NAME
config_to_save['num_train_examples'] = len(train_examples)
config_to_save['num_val_examples'] = len(val_examples)

with open(OUTPUT_DIR / "training_config.json", 'w') as f:
    json.dump(config_to_save, f, indent=2)

print(f"✓ Configuración final guardada en {OUTPUT_DIR / 'training_config.json'}")


✓ Configuración final guardada en 2_Modelos/clustering/training_config.json


## 5. Generación de Embeddings

In [19]:
texts = df['text'].tolist()

# Generar embeddings en batches
embeddings = model.encode(
    texts,
    batch_size=config['batch_size'],
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True # CLAVE: Normalizar para mejor clustering
)

print(f"✓ Embeddings generados: {embeddings.shape}")
np.save(OUTPUT_DIR / "embeddings.npy", embeddings)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

✓ Embeddings generados: (434, 384)


In [20]:
# HDBSCAN (clustering density-based)
clusterer = HDBSCAN(
    min_cluster_size=config['min_cluster_size'],
    min_samples=config['min_samples'],
    metric='euclidean',
    cluster_selection_method='eom'
)

cluster_labels = clusterer.fit_predict(embeddings)

# Estadísticas
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

print(f"\n✓ Clusters encontrados: {n_clusters}")
print(f"✓ Puntos outlier: {n_noise} ({n_noise/len(cluster_labels)*100:.1f}%)")

# Distribución
cluster_counts = Counter(cluster_labels)
print(f"\nDistribución de clusters:")
for cluster_id, count in sorted(cluster_counts.items()):
    if cluster_id != -1:
        print(f"  Cluster {cluster_id}: {count} documentos")

np.save(OUTPUT_DIR / "cluster_labels.npy", cluster_labels)


✓ Clusters encontrados: 4
✓ Puntos outlier: 179 (41.2%)

Distribución de clusters:
  Cluster 0: 5 documentos
  Cluster 1: 232 documentos
  Cluster 2: 12 documentos
  Cluster 3: 6 documentos


## 7. Evaluación de resultados

In [21]:

# Filtrar outliers (-1) para las métricas
mask = cluster_labels != -1
filtered_embeddings = embeddings[mask]
filtered_labels = cluster_labels[mask]

metrics = {}

if len(set(filtered_labels)) >= 2:
    # Silhouette Score
    silhouette = silhouette_score(filtered_embeddings, filtered_labels)
    print(f"\nSilhouette Score: {silhouette:.4f}")
    
    # Davies-Bouldin Index
    davies_bouldin = davies_bouldin_score(filtered_embeddings, filtered_labels)
    print(f"\nDavies-Bouldin Index: {davies_bouldin:.4f}")
    
    metrics = {
        'silhouette_score': float(silhouette),
        'davies_bouldin_index': float(davies_bouldin),
        'n_clusters': len(set(filtered_labels)),
        'n_outliers': int((cluster_labels == -1).sum()),
        'outlier_percentage': float((cluster_labels == -1).sum() / len(cluster_labels) * 100)
    }
    
    with open(OUTPUT_DIR / "clustering_metrics.json", 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print(f"\n✓ Métricas guardadas en {OUTPUT_DIR / 'clustering_metrics.json'}")

else:
    print("⚠ Muy pocos clusters para calcular métricas (Se necesita al menos 2 clusters).")




Silhouette Score: 0.0517

Davies-Bouldin Index: 2.1143

✓ Métricas guardadas en 2_Modelos/clustering/clustering_metrics.json


## 8. Exportación de resultados

In [22]:
pca_export = PCA(n_components=2, random_state=config['random_seed'])
embeddings_2d_export = pca_export.fit_transform(embeddings)

# Crear DataFrame de resultados
df_results = df.copy()
df_results['cluster_label'] = cluster_labels
df_results['pca_dim_1'] = embeddings_2d_export[:, 0]
df_results['pca_dim_2'] = embeddings_2d_export[:, 1]

# Definir la ruta del archivo de salida
output_csv_path = OUTPUT_DIR / "clustering_results_consolidated.csv"

# Exportar a CSV
df_results.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"✓ Resultados consolidados exportados con éxito a:")
print(f"  -> {output_csv_path}")
print("Columnas: [text, title, source, cluster_label, pca_dim_1, pca_dim_2]")


✓ Resultados consolidados exportados con éxito a:
  -> 2_Modelos/clustering/clustering_results_consolidated.csv
Columnas: [text, title, source, cluster_label, pca_dim_1, pca_dim_2]
